In [1]:
import sys

assert sys.version_info >= (3, 10)
IS_COLAB = "google.colab" in sys.modules


In [2]:
if IS_COLAB:
    !git clone https://github.com/stachuapa123/ASR_project.git

    # 2. Wejście do folderu projektu
    %cd ASR_project

    # 3. Zmiana gałęzi na 'PierwszyModel' (bo tam pracujesz)
    !git checkout Augmenting

    !pip install torchmetrics

In [3]:
%load_ext autoreload
%autoreload 2

import os, glob
import numpy as np
import torch
import torch.nn as nn
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.io import wavfile
from src.constants import Constants as C
from pathlib import Path
import torchmetrics

In [5]:
from src.augment import augment_audio, SpecAugment
from src.parsers import parse_phonemes, wav_to_logmel, windows_and_labels, build_audio_cache, build_augmented_cache
from src.parsers import PhonemeWindowDataset
from src.parsers import AugmentedCacheDataset, DoubledAugmentedCacheDataset
from src.NeuralModel import CRNN
from src.trainers import train_model, evaluate_tm, save_checkpoint



In [6]:
if (IS_COLAB):
    from google.colab import drive
    drive.mount('/content/drive')



In [7]:
DATA_DIR2 = "../AutorskieDane/AutorskiDataset"

if not os.path.exists(DATA_DIR2) and IS_COLAB:
    !cp -r '/content/drive/MyDrive/AutorskiDataset' /content/

CACHE_PATH = '/content/dataset_cache_aug.pt'

if(IS_COLAB):
    DRIVE_CACHE = '/content/drive/MyDrive/dataset_cache_aug.pt'
    CACHE_PATH = '/content/dataset_cache_aug.pt'
else:
    DRIVE_CACHE = '../dataset_cache_aug.pt'
    CACHE_PATH = '../AugmentCaches/dataset_cache_aug.pt'

In [8]:
if os.path.exists(DRIVE_CACHE):
    !cp $DRIVE_CACHE $CACHE_PATH
    print('cache wczytany z Drive')
elif not os.path.exists(CACHE_PATH):
    print('budowanie cache (jednorazowe)...')
    build_augmented_cache(                            # ← właściwa funkcja
        data_dir=DATA_DIR2,
        output_path=CACHE_PATH,
        n_augmentations=6,
        standardize=True,
        silences_same=True,
    )
    !cp $CACHE_PATH $DRIVE_CACHE

In [9]:
data = torch.load(CACHE_PATH, weights_only=True)
X = data['X']           # (n_windows, n_aug, n_mels, win_frames)
y = data['y']
n_aug = data['n_aug']
print(f'X={X.shape}, y={y.shape}, n_aug={n_aug}')

X=torch.Size([45035, 6, 128, 8]), y=torch.Size([45035]), n_aug=6


In [10]:
generator = torch.Generator().manual_seed(0)
indices = torch.randperm(len(X), generator=generator)
n_val = int(0.2 * len(X))
train_idx, val_idx = indices[n_val:], indices[:n_val]
train_set = DoubledAugmentedCacheDataset(X[train_idx], y[train_idx], train=True)

val_set = AugmentedCacheDataset(X[val_idx], y[val_idx], train=False)
mel_augmenter = SpecAugment(freq_mask_percent=0.1, time_mask_percent=0.125, p=0.5)

In [11]:
train_loader = DataLoader(train_set, batch_size=512, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=1024, shuffle=False,
                          num_workers=2, pin_memory=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CRNN().to(device)
print('params:', sum(p.numel() for p in model.parameters()))

params: 1252103


In [12]:
optim = torch.optim.NAdam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss(label_smoothing=0.1)
EPOCHS = 4
metric = torchmetrics.Accuracy(task='multiclass', num_classes=C.N_CLASSES).to(device)

history = train_model(
    model, optim, crit, metric,
    train_loader, val_loader,
    n_epochs=EPOCHS,
    patience=2, factor=0.5,
    device=device,
    spec_augment=mel_augmenter,
)

c:\Users\mmapa\Desktop\Eti\ASR\ASR_project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1/4  batch 35/141  loss=2.6108

KeyboardInterrupt: 

In [ ]:
CHECKPOINT_PATH = "../trained_models/Augmenty2DlaAutorskich.pt"

config = {
    "sample_rate": C.SAMPLE_RATE,
    "n_fft": C.N_FFT,
    "hop_length": C.HOP_LENGTH,
    "n_mels": C.N_MELS,
    "win_frames": C.WIN_FRAMES,
    "shift_frames": C.SHIFT_FRAMES,
    "hidden": 64,
}

save_checkpoint(
    CHECKPOINT_PATH,
    model,
    labels=C.LABELS,
    config=config,
    history=history,
)
print(f"saved to {CHECKPOINT_PATH}")

saved to ../trained_models/Augmenty1DlaAutorskich.pt


/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:245: WavFileWarning: Chunk (non-data) not understood, skipping it.
  audio = audio.astype(np.float32) / np.iinfo(audio.dtype).max
/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:245: WavFileWarning: Chunk (non-data) not understood, skipping it.
  audio = audio.astype(np.float32) / np.iinfo(audio.dtype).max
/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:245: WavFileWarning: Chunk (non-data) not understood, skipping it.
  audio = audio.astype(np.float32) / np.iinfo(audio.dtype).max
/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:245: WavFileWarning: Chunk (non-data) not understood, skipping it.
  audio = audio.astype(np.float32) / np.iinfo(audio.dtype).max
/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:245: WavFileWarning: Chunk (non-data) not understood, skipping it.
  audio = audio.astype(np.float32) / np.iinfo(audio.dtype).max
/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:24